In [ ]:
import cv2, glob, os
import numpy as np
from collections import Counter
from google.colab import drive

drive.mount('/content/drive')


PASTA   = "/content/drive/MyDrive/Doutorado/Robot Perception/Imagens"
SQUARE  = 15.0                 # mm
PADROES = [(7, 10), (10, 7)]   # cantos internos

arquivos = sorted(glob.glob(PASTA + "/*.[jJ][pP][gG]") + glob.glob(PASTA + "/*.[pP][nN][gG]"))
print(len(arquivos), "imagens encontradas")

crit  = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
flags = cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_NORMALIZE_IMAGE

dets = []
for f in arquivos:
    gray = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2GRAY)
    s = min(1.0, 1600 / max(gray.shape))
    small = cv2.resize(gray, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    for p in PADROES:
        ok, c = cv2.findChessboardCorners(small, p, flags=flags)
        if ok:
            c = cv2.cornerSubPix(gray, (c / s).astype(np.float32), (11, 11), (-1, -1), crit)
            objp = np.zeros((p[0]*p[1], 3), np.float32)
            objp[:, :2] = np.mgrid[0:p[0], 0:p[1]].T.reshape(-1, 2) * SQUARE
            dets.append((os.path.basename(f), gray.shape[::-1], objp, c))
            break
    else:
        print("sem detecção:", os.path.basename(f))

#
print("Resoluções encontradas:", Counter(d[1] for d in dets))
tamanho = Counter(d[1] for d in dets).most_common(1)[0][0]
dets = [d for d in dets if d[1] == tamanho]

nomes     = [d[0] for d in dets]
objpoints = [d[2] for d in dets]
imgpoints = [d[3] for d in dets]
print(len(dets), "imagens usadas na calibração")
assert len(dets) >= 5, "Poucas imagens válidas."

#Calibração
rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, tamanho, None, None)

#Erro de reprojeção
erros = []
for i in range(len(dets)):
    proj, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], K, dist)
    erros.append(np.linalg.norm(imgpoints[i].reshape(-1, 2) - proj.reshape(-1, 2), axis=1).mean())

#Resultados
np.set_printoptions(precision=4, suppress=True)
print("\nResolução:", tamanho)
print("Erro RMS (px):", round(rms, 4))
print("Erro médio de reprojeção (px):", round(float(np.mean(erros)), 4))
print("Matriz K:\n", K)
print("Distorção (k1, k2, p1, p2, k3):", dist.ravel())